# This is an interactive notebook to analyze trends in video game sales data from 2016 to make marketing decision in 2017

### The goal of this dataset is to find out trends of patterns of profitable games in order to see which games to invest a marketing budget into 

#### The start of the EDA will do some data preprocessing including replacing or dropping null values, confirming the abscene of duplicates, and making sure all types make sense

In [2]:
import pandas as pd 
import math as m
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import plotly.express as px

In [3]:
df = pd.read_csv('games.csv')

In [4]:
display(df.head())

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


### Convert all columns to lowercase

In [5]:
df.columns = df.columns.str.lower()

In [6]:
display(df.head())

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [7]:
display(df.sample(10))

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
4460,SpongeBob's Atlantis SquarePantis,PS2,2007.0,Action,0.36,0.01,0.00,0.06,NaN,tbd,E
3022,Fossil Fighters: Champions,DS,2010.0,Role-Playing,0.31,0.00,0.34,0.02,68.0,7.1,E
12228,Terraria,PS4,2014.0,Action,0.00,0.05,0.01,0.01,83.0,7.9,T
979,Disney's The Lion King,SNES,1994.0,Platform,1.26,0.39,0.08,0.06,NaN,NaN,NaN
7307,Teenage Mutant Ninja Turtles: Smash-Up,PS2,2009.0,Fighting,0.11,0.08,0.00,0.03,NaN,8.4,E10+
4096,Mario Golf: World Tour,3DS,2014.0,Sports,0.15,0.13,0.17,0.03,78.0,8.2,E
9981,Where the Wild Things Are,Wii,2009.0,Platform,0.11,0.00,0.00,0.01,65.0,tbd,E10+
5948,Guardian Heroes,SAT,1995.0,Role-Playing,0.00,0.00,0.29,0.00,NaN,NaN,NaN
4635,NHL Slapshot,Wii,NaN,Sports,0.39,0.00,0.00,0.02,76.0,8.1,E
7738,Conan,X360,2007.0,Action,0.16,0.02,0.00,0.01,69.0,7.3,M


In [8]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16715 non-null  object 
 2   year_of_release  16446 non-null  float64
 3   genre            16713 non-null  object 
 4   na_sales         16715 non-null  float64
 5   eu_sales         16715 non-null  float64
 6   jp_sales         16715 non-null  float64
 7   other_sales      16715 non-null  float64
 8   critic_score     8137 non-null   float64
 9   user_score       10014 non-null  object 
 10  rating           9949 non-null   object 
dtypes: float64(6), object(5)
memory usage: 1.4+ MB
None


### The only types I need to convert are year_of_release and user_score ... everything else seems to make sense 
##### making year_of relase an int made it cleaner and user_score being a float will be better for calculating later ... I wanted to convert to np.nan also so it was clear how many null values I have

In [9]:
# Change from floar to int
df['year_of_release'] = df['year_of_release'].astype('Int64')

# Change from object to float replace tbd and NaN with np.nan
df['user_score'] = df['user_score'].replace(['NaN', 'tbd'], np.nan)
df['user_score'] = df['user_score'].astype(float)

### Checking for missing values and duplicates 

In [10]:
print(df.isna().sum())
print(df.duplicated().sum())

name                  2
platform              0
year_of_release     269
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8578
user_score         9125
rating             6766
dtype: int64
0


### No fully duplicate rows but there is null values ... lots of missing values for both user and critic score and the rating ... and some missing year_of_release data
##### Could be from games not being popular enough for a review or from data collection issues

##### For both name and genre there is only two missing rows so I will just drop these rows
##### For rating I think there is too many missing to drop them and taking an average for a certain type might cause problems down the line so I think it's best to just replace them with Unknown

In [11]:
# drop 2 missing rows from name and genre
df = df.dropna(subset=['name', 'genre'])

# replace null with unknown in rating column
df['rating'] = df['rating'].fillna('Unknown')


### For Titles that repeat across platform I'm going to take the mode for year_of_release and apply it to the null values

In [12]:
df['year_of_release'] = df.groupby('name')['year_of_release'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
print(df['year_of_release'].isna().sum())


146


#### There is still 146 missing year_of_release rows... at this point I think I can just drop those rows as I don't want it interfering with calculations later 

In [13]:
df = df.dropna(subset=['year_of_release'])
print(df['year_of_release'].isna().sum())

0


### Making a column to see if a given game has a critic score or user score ... I will use this later to compare calclations between games that have score and those that do not 

In [14]:
df['has_critic_score'] = df['critic_score'].notna().astype(int)
df['has_user_score'] = df['user_score'].notna().astype(int)

In [15]:
display(df.head())

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,has_critic_score,has_user_score
0,Wii Sports,Wii,2006,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E,1,1
1,Super Mario Bros.,NES,1985,Platform,29.08,3.58,6.81,0.77,NaN,NaN,Unknown,0,0
2,Mario Kart Wii,Wii,2008,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E,1,1
3,Wii Sports Resort,Wii,2009,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E,1,1
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,Unknown,0,0


### Calculate total sales for each game

In [16]:
# total sales
df['total_sales'] = df[['na_sales', 'eu_sales', 'jp_sales', 'other_sales']].sum(axis=1)
# make sure it's a float
df['total_sales'] = df['total_sales'].astype(float)

In [17]:
display(df.head())

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,has_critic_score,has_user_score,total_sales
0,Wii Sports,Wii,2006,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E,1,1,82.54
1,Super Mario Bros.,NES,1985,Platform,29.08,3.58,6.81,0.77,NaN,NaN,Unknown,0,0,40.24
2,Mario Kart Wii,Wii,2008,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E,1,1,35.52
3,Wii Sports Resort,Wii,2009,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E,1,1,32.77
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,Unknown,0,0,31.38


In [18]:
#grouping by year of release and counting the number of games released each year
games_released_year = df.groupby('year_of_release')['name'].count().reset_index()

#simple line plot using plotly express
#x-axis is year_of_release and y-axis is the count of games released
fig = px.line(games_released_year, x='year_of_release', y='name', title='Games released per year')

fig.update_layout(xaxis_title='Year of Release', yaxis_title='Number of Games')
fig.show()


### plot showing the volume of games released per year from the dataset

### I want to claculate the trends while also including types of platforms ... There is 31 different platforms which would be hard to graph so I will just do the top 10 for imporved clarity

In [19]:
print(df['platform'].value_counts().head(31))
print(df['platform'].nunique())

top_ten_platforms = df['platform'].value_counts().head(10).index.tolist()
# filter the dataframe to include only the top 10 platforms
df_top_ten = df[df['platform'].isin(top_ten_platforms)]


platform
PS2     2140
DS      2129
PS3     1320
Wii     1301
X360    1250
PSP     1203
PS      1190
PC       970
XB       817
GBA      813
GC       552
3DS      515
PSV      429
PS4      392
N64      318
XOne     247
SNES     239
SAT      173
WiiU     147
2600     117
NES       98
GB        97
DC        52
GEN       27
NG        12
SCD        6
WS         6
3DO        3
TG16       2
GG         1
PCFX       1
Name: count, dtype: int64
31


In [20]:
games_released_year_top_ten_platform = df_top_ten.groupby(['year_of_release', 'platform'])['name'].count().reset_index()

#simple line plot using plotly express
fig = px.line(games_released_year_top_ten_platform, x='year_of_release', y='name', color='platform', title='Games released per year by platform')

fig.update_layout(xaxis_title='Year of Release', yaxis_title='Number of Games')
fig.show()

### This graph shows the games relesed per year by platform for the top ten platforms ... this shows some of these platforms whicha re the most popular overall aren't active anymore which might influence our decision making in the year '2017' 

#### Also the graph shows that the lifespan of one of the major platforms is around 8 years

### I think to get an accurate assement of the modern game market it would be best to focus on the current decade from 2011 on because the data also has lots of old platforms that are being phased out 

In [21]:
df = df[df['year_of_release'] >= 2011]

In [22]:
# Confirmation of the filtering
print(df['year_of_release'].min())
print(df['year_of_release'].max())

# Check for the new total new amount of platforms 
print(df['platform'].nunique())

2011
2016
12


In [23]:
top_ten_platforms = df['platform'].value_counts().head(10).index.tolist()
# filter the dataframe to include only the top 10 platforms
df_top_ten = df[df['platform'].isin(top_ten_platforms)]

games_released_year_top_ten_platform = df_top_ten.groupby(['year_of_release', 'platform'])['name'].count().reset_index()
#simple line plot using plotly express
fig = px.line(games_released_year_top_ten_platform, x='year_of_release', y='name', color='platform', title='Games released per year by platform')

fig.update_layout(xaxis_title='Year of Release', yaxis_title='Number of Games')
fig.show()

### This graph is showing all platforms are actively decreasing total volume of games created except PS4 and Xone are growing and PC is stayying roughly the same

In [24]:
total_sales_platform = df.groupby('platform')['total_sales'].sum().reset_index()

top_ten_platforms = total_sales_platform.nlargest(10, 'total_sales')['platform'].tolist()
# filter the dataframe to include only the top 10 platforms
df_top_ten = df[df['platform'].isin(top_ten_platforms)]

total_sales_platform = df_top_ten.groupby(['platform', 'year_of_release'])['total_sales'].sum().reset_index()
#simple line plot using plotly express
fig = px.line(total_sales_platform, x='year_of_release', y='total_sales', color='platform', title='Total sales by platform')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='Total Sales')
fig.show()

In [30]:
# box plot for total sales by platform 
fig = px.box(df, x='platform', y='total_sales', title='Total sales by platform')
fig.update_layout(xaxis_title='Platform', yaxis_title='Total Sales')
fig.show()
fig = px.box(df, x='platform', y='total_sales', title='Total sales by platform')
fig.update_layout(xaxis_title='Platform', yaxis_title='Total Sales')
fig.update_yaxes(range=[0, 2])
fig.show()



### this graph shows total_sales for each platform and clearly shows that the PS4 and Xbox One are by far the leading two platforms in terms of total sales in $ 

### the median for PS4 and Xbox One are about the same but the PS4 has a lot more games that have extremely high sales

### I added another graph with a ylimit of two million to more accurately represent the average amount from each platform 

In [ ]:
# scatter plot for correlation between reviews and sales exclduing NaN values
df_clean = df.dropna(subset=['critic_score', 'user_score', 'total_sales'])

# Step 2: Get top 4 platforms by number of games
top_platforms = df_clean['platform'].value_counts().nlargest(4).index

# Step 3: Loop through each platform
for platform in top_platforms:
    subset = df_clean[df_clean['platform'] == platform]

    # Pearson correlation for critic_score
    corr_critic, _ = stats.pearsonr(subset['critic_score'], subset['total_sales'])
    fig = px.scatter(subset, x='critic_score', y='total_sales',
                     title=f'{platform}: Critic Score vs Total Sales<br>Pearson r = {corr_critic:.2f}')
    fig.update_layout(xaxis_title='Critic Score', yaxis_title='Total Sales')
    fig.show()

    # Pearson correlation for user_score
    corr_user, _ = stats.pearsonr(subset['user_score'], subset['total_sales'])
    fig = px.scatter(subset, x='user_score', y='total_sales',
                     title=f'{platform}: User Score vs Total Sales<br>Pearson r = {corr_user:.2f}')
    fig.update_layout(xaxis_title='User Score', yaxis_title='Total Sales')
    fig.show()


### The scattrer plot's show total sales and user/critic score per genre ... The graphs / pearson correlation claculation show there is a decent correlation between critic score and total sales for pretty much every platform... Although there is almost no correlation between user score and total sales for all platforms

In [ ]:
# Optional: filter to games released on more than one platform
multi_platform_games = df.groupby('name')['platform'].nunique()
multi_platform_games = multi_platform_games[multi_platform_games > 1].index

multi_platform_df = df[df['name'].isin(multi_platform_games)]

# Optional: focus on top-selling games
top_games = multi_platform_df.groupby('name')['total_sales'].sum().nlargest(12).index
top_df = multi_platform_df[multi_platform_df['name'].isin(top_games)]

# Plot
fig = px.bar(top_df,
             x='name',
             y='total_sales',
             color='platform',
             barmode='group',
             title='Top 12 Multi-Platform Games by Total Sales',
             labels={'total_sales': 'Total Sales (millions)', 'name': 'Game Title'},
             height=600)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

### This graph shows for the 12 most popular games PS4 tends to edge out XboxOne 

In [34]:
total_sales_genre = df.groupby('genre')['total_sales'].mean().reset_index()
# sort the values in descending order
total_sales_genre = total_sales_genre.sort_values(by='total_sales', ascending=False)
# simple bar plot using plotly express
fig = px.bar(total_sales_genre, x='genre', y='total_sales', title='Mean total sales by genre')
fig.update_layout(xaxis_title='Genre', yaxis_title='Total Sales')
fig.show()

### I calculated the mean sales for each genre in each genre and shooters are almost double the runner up platform ... 

In [46]:
# top five platforms for each region na sales, eu sales, jp sales, other sales
top_five_platforms = df.groupby('platform')['na_sales'].sum().nlargest(5).index.tolist()
# filter the dataframe to include only the top 5 platforms
df_top_five = df[df['platform'].isin(top_five_platforms)]
# group by platform and year of release and sum the sales
na_sales_platform = df_top_five.groupby(['platform', 'year_of_release'])['na_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(na_sales_platform, x='year_of_release', y='na_sales', color='platform', title='NA Sales by platform')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='NA Sales')
fig.show()
# top five platforms for each region eu sales, jp sales, other sales
jp_sales_platform = df_top_five.groupby(['platform', 'year_of_release'])['jp_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(jp_sales_platform, x='year_of_release', y='jp_sales', color='platform', title='JP Sales by platform')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='JP Sales')
fig.show()
# top five platforms for eu sales 
eu_sales_platform = df_top_five.groupby(['platform', 'year_of_release'])['eu_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(eu_sales_platform, x='year_of_release', y='eu_sales', color='platform', title='EU Sales by platform')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='EU Sales')
fig.show()
# top five platforms for other sales
other_sales_platform = df_top_five.groupby(['platform', 'year_of_release'])['other_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(other_sales_platform, x='year_of_release', y='other_sales', color='platform', title='Other Sales by platform')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='Other Sales')
fig.show() 


### These graphs show that in all regions except NA PS4 sales are considerably higher than Xbox One ... IN japan Xbox One sales are non-existent

In [47]:
# box plot of top 5 genres per region 
top_five_genres = df.groupby('genre')['na_sales'].sum().nlargest(5).index.tolist()
# filter the dataframe to include only the top 5 genres
df_top_five = df[df['genre'].isin(top_five_genres)]
# group by genre and year of release and sum the sales
na_sales_genre = df_top_five.groupby(['genre', 'year_of_release'])['na_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(na_sales_genre, x='year_of_release', y='na_sales', color='genre', title='NA Sales by genre')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='NA Sales')
fig.show()
# top five genres for eu sales
eu_sales_genre = df_top_five.groupby(['genre', 'year_of_release'])['eu_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(eu_sales_genre, x='year_of_release', y='eu_sales', color='genre', title='EU Sales by genre')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='EU Sales')
fig.show()
# top five genres for jp sales
jp_sales_genre = df_top_five.groupby(['genre', 'year_of_release'])['jp_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(jp_sales_genre, x='year_of_release', y='jp_sales', color='genre', title='JP Sales by genre')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='JP Sales')
fig.show()
# top five genres for other sales
other_sales_genre = df_top_five.groupby(['genre', 'year_of_release'])['other_sales'].sum().reset_index()
# simple line plot using plotly express
fig = px.line(other_sales_genre, x='year_of_release', y='other_sales', color='genre', title='Other Sales by genre')
fig.update_layout(xaxis_title='Year of Release', yaxis_title='Other Sales')
fig.show()


### These graphs show that there is still a correlation to the overall market where Shooters and action tend to be more than other genres except in Japan where role-playing games are also just as popualr as the other two

In [49]:
# ESRB rating affecting sales   
ratings_sales = df.groupby('rating')['total_sales'].sum().reset_index()
ratings_sales = ratings_sales.sort_values(by='total_sales', ascending=False)
# simple bar plot using plotly express
fig = px.bar(ratings_sales, x='rating', y='total_sales', title='Total sales by rating')
fig.update_layout(xaxis_title='Rating', yaxis_title='Total Sales')
fig.show()


#### NA 
rating_sales_na = df.groupby('rating')['na_sales'].sum().reset_index()
rating_sales_na = rating_sales_na.sort_values(by='na_sales', ascending=False)
# simple bar plot using plotly express
fig = px.bar(rating_sales_na, x='rating', y='na_sales', title='NA Sales by rating')
fig.update_layout(xaxis_title='Rating', yaxis_title='NA Sales')
fig.show()

#### EU 
rating_sales_eu = df.groupby('rating')['eu_sales'].sum().reset_index()
rating_sales_eu = rating_sales_eu.sort_values(by='eu_sales', ascending=False)
# simple bar plot using plotly express
fig = px.bar(rating_sales_eu, x='rating', y='eu_sales', title='EU Sales by rating')
fig.update_layout(xaxis_title='Rating', yaxis_title='EU Sales')
fig.show()

#### JP
rating_sales_jp = df.groupby('rating')['jp_sales'].sum().reset_index()
rating_sales_jp = rating_sales_jp.sort_values(by='jp_sales', ascending=False)
# simple bar plot using plotly express
fig = px.bar(rating_sales_jp, x='rating', y='jp_sales', title='JP Sales by rating')
fig.update_layout(xaxis_title='Rating', yaxis_title='JP Sales')
fig.show()

#### Other
rating_sales_other = df.groupby('rating')['other_sales'].sum().reset_index()
rating_sales_other = rating_sales_other.sort_values(by='other_sales', ascending=False)
# simple bar plot using plotly express
fig = px.bar(rating_sales_other, x='rating', y='other_sales', title='Other Sales by rating')
fig.update_layout(xaxis_title='Rating', yaxis_title='Other Sales')
fig.show()


### The following graphs shows that the M rating for games is by far the most popular except for in Japan where it is the 3rd most popular type of genre

### Null and alternattive Hypothesis for testing average user ratings are the same for Xbox One and PC Games 
##### I'm using a two tailed test because we're looking for any difference ... The Results show There isn't a significant difference between the review of each platform
### Null Hypothesis :
##### there is no difference in average user ratings between Xbox One and PC games.


### Alternative Hypothesis 
##### There is a difference in average user ratings between Xbox One and PC games.




In [ ]:
xbox = df[df['platform'] == 'XOne']['user_score'].dropna()
pc = df[df['platform'] == 'PC']['user_score'].dropna()

# Ttest
t_stat, p_val = stats.ttest_ind(xbox, pc, equal_var=False)

alpha = 0.05

print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4f}")
if p_val < alpha:
    print("Reject the null hypothesis: user ratings are significantly different.")
else:
    print("Fail to reject the null hypothesis: no significant difference.")

T-statistic: 0.3718, P-value: 0.7102
Fail to reject the null hypothesis: no significant difference.


### Null and alternative hypothesis testing for user rtatings between action and sports games
##### Another two tailed test becasue we're looking for any difference ... The results show the ratings differ significantly especially compare to the previous test
### Null Hypothesis:
##### There is no difference in average user ratings between Action and Sports games.


### Alternative Hypothesis:
##### There is a difference in average user ratings between Action and Sports games.


In [51]:
action = df[df['genre'] == 'Action']['user_score'].dropna()
sports = df[df['genre'] == 'Sports']['user_score'].dropna()

# Welch’s t-test again
t_stat, p_val = stats.ttest_ind(action, sports, equal_var=False)

alpha = 0.05

print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4f}")
if p_val < alpha:
    print("Reject the null hypothesis: user ratings are significantly different.")
else:
    print("Fail to reject the null hypothesis: no significant difference.")

T-statistic: 7.5664, P-value: 0.0000
Reject the null hypothesis: user ratings are significantly different.


# Conclusion 
### 1. The null hypothesis fails to be rejected for the comparison between Xbox One and PC user ratings, indicating a significant correlation in user ratings between these two platforms.
### 2. The null hypothesis is rejected for the comparison between Action and Sports genres, indicating a significant difference in user ratings between these two genres.

### The ideal candidate for a game would be a game that is released on multiple platforms, has a high critic score, and is in the shooter genre.
### If it were to only be released on one platform, the ideal platform would be the Play Station 4, as it has the highest total sales But the xbox would be okay as there is much less of a volume of games to compete with.
### Ideally it is released to both consoles
### For focus on the marketing strategy, the ideal region to focus on would be North America, as it has the highest total sales.
### We can also spend less money marketing in Japan as their relative sales are significantly lower than the other regions for Action or Shooters and the overall market is smaller. 
### It would be ideal for the game to be rated M 